# 🔀 Notebook 3 · Seq2Seq: de una secuencia a otra

**Diplomado CENIA · ¿Cómo lee una red neuronal?** · Acto 3 de la clase

1. Una tarea de juguete (invertir secuencias) para ver el encoder–decoder por dentro.
2. Entrenar un **encoder–decoder sin atención** y **medir el cuello de botella**.
3. Programar la **atención** y ver dónde "mira" el decoder.
4. **Traducir español → inglés** de verdad, con el mismo modelo.

> 💡 Activa la GPU en **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** (también funciona en CPU, un poco más lento).

In [1]:
import re, time, math, random
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(0); random.seed(0)
print("Usando:", device)

Usando: cuda


## 1. Una tarea de juguete: invertir secuencias

Antes de traducir idiomas, usemos una tarea donde **sabemos la respuesta** y que entrena rápido:

`3 1 4 1 5`  →  `5 1 4 1 3`

Es perfecta para ver el **cuello de botella**: para invertir, el decoder tiene que escribir primero lo que el encoder leyó **al final**… y al último lo que leyó **al principio**, que es lo que más se diluye en el vector de contexto.

Usamos 3 tokens especiales, igual que en la clase: `[PAD]` (relleno para armar batches), `<BOS>` (empieza a escribir) y `<EOS>` (terminé).

In [2]:
PAD, BOS, EOS = 0, 1, 2             # tokens especiales (los mismos en todo el notebook)
VOCAB_JUGUETE = 13                  # 3 especiales + los dígitos 0–9 (el dígito d es el token d + 3)

def generar_batch(n, largo_min=3, largo_max=25):
    largos = torch.randint(largo_min, largo_max + 1, (n,))
    M = int(largos.max())
    src = torch.full((n, M), PAD)
    tgt = torch.full((n, M + 2), PAD)
    for i, L in enumerate(largos.tolist()):
        s = torch.randint(3, 13, (L,))      # dígitos 0–9 → tokens 3–12
        src[i, :L] = s
        tgt[i, 0] = BOS
        tgt[i, 1:L + 1] = s.flip(0)       # la respuesta: la secuencia invertida
        tgt[i, L + 1] = EOS
    return src.to(device), tgt.to(device), largos.to(device)

def mostrar(ids):
    nombres = {PAD: "·", BOS: "<BOS>", EOS: "<EOS>"}
    return " ".join(nombres.get(i, str(i - 3)) for i in ids)

src, tgt, largos = generar_batch(3, 4, 6)
for s, t in zip(src.tolist(), tgt.tolist()):
    print(f"{mostrar(s):14s} →  {mostrar(t)}")

0 3 9 7 3 7    →  <BOS> 7 3 7 9 3 0 <EOS>
3 1 6 6 · ·    →  <BOS> 6 6 1 3 <EOS> · ·
9 8 6 6 8 4    →  <BOS> 4 8 6 6 8 9 <EOS>


## 2. Encoder–decoder sin atención

Tres piezas, igual que en la diapositiva:

1. **Encoder** (GRU): lee la entrada y su **última memoria** es el vector de contexto, el "post-it".
2. **Decoder** (GRU): parte desde ese contexto y escribe de a un token. En cada paso recibe el token anterior.
3. **Teacher forcing**: al *entrenar*, el decoder recibe el token **correcto** anterior (no el que él predijo). Al *generar*, recibe el suyo.

La clase `Decoder` ya trae la opción de atención (la definimos en la sección 3); por ahora la dejamos apagada (`atencion=False`).

In [3]:
class Encoder(nn.Module):
    def __init__(self, vocab, d_emb=32, d_h=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_emb, padding_idx=PAD)
        self.rnn = nn.GRU(d_emb, d_h, batch_first=True)

    def forward(self, src, largos):
        salidas, _ = self.rnn(self.emb(src))                      # memoria en cada paso
        contexto = salidas[torch.arange(len(largos)), largos - 1]  # última memoria REAL (sin contar [PAD])
        return salidas, contexto


class Decoder(nn.Module):
    def __init__(self, vocab, d_emb=32, d_h=128, atencion=False):
        super().__init__()
        self.atencion = AtencionBahdanau(d_h) if atencion else None
        self.emb = nn.Embedding(vocab, d_emb, padding_idx=PAD)
        self.rnn = nn.GRUCell(d_emb + (d_h if atencion else 0), d_h)
        self.salida = nn.Linear(d_h, vocab)

    def paso(self, y_anterior, h, enc_salidas, mascara):
        e = self.emb(y_anterior)
        pesos = None
        if self.atencion is not None:
            contexto, pesos = self.atencion(h, enc_salidas, mascara)
            e = torch.cat([e, contexto], dim=-1)     # el decoder ve el token anterior + lo que "miró"
        h = self.rnn(e, h)
        return self.salida(h), h, pesos


class Seq2Seq(nn.Module):
    def __init__(self, vocab_src, vocab_tgt, atencion=False, d_emb=32, d_h=128):
        super().__init__()
        self.enc = Encoder(vocab_src, d_emb=d_emb, d_h=d_h)
        self.dec = Decoder(vocab_tgt, d_emb=d_emb, d_h=d_h, atencion=atencion)

    def forward(self, src, largos, tgt):
        """Entrenamiento, con teacher forcing."""
        enc_salidas, h = self.enc(src, largos)
        mascara = src != PAD
        logits = []
        for t in range(tgt.size(1) - 1):
            out, h, _ = self.dec.paso(tgt[:, t], h, enc_salidas, mascara)   # ← token CORRECTO anterior
            logits.append(out)
        return torch.stack(logits, dim=1)

    @torch.no_grad()
    def generar(self, src, largos, max_largo=30):
        """Generación greedy: cada token predicho entra como input del paso siguiente."""
        enc_salidas, h = self.enc(src, largos)
        mascara = src != PAD
        y = torch.full((src.size(0),), BOS, device=src.device)
        tokens, todos_pesos = [], []
        for _ in range(max_largo):
            out, h, pesos = self.dec.paso(y, h, enc_salidas, mascara)
            y = out.argmax(dim=-1)                                            # ← su PROPIA predicción
            tokens.append(y)
            todos_pesos.append(pesos)
        pesos = torch.stack(todos_pesos, 1) if todos_pesos[0] is not None else None
        return torch.stack(tokens, dim=1), pesos

In [ ]:
def entrenar_juguete(modelo, pasos=1500, cada=300):
    opt = torch.optim.Adam(modelo.parameters(), lr=3e-3)
    inicio = time.time()
    for paso in range(1, pasos + 1):
        src, tgt, largos = generar_batch(64)
        logits = modelo(src, largos, tgt)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1), ignore_index=PAD)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        opt.step()
        if paso % cada == 0:
            print(f"paso {paso:5d} · loss {loss.item():.3f} · {time.time() - inicio:.0f}s")
    return modelo


def precision_por_largo(modelo, largos_eval=(5, 10, 15, 20, 25)):
    modelo.eval()
    resultado = {}
    for L in largos_eval:
        src, tgt, largos = generar_batch(300, L, L)
        pred, _ = modelo.generar(src, largos, max_largo=L)
        resultado[L] = (pred == tgt[:, 1:L + 1]).float().mean().item()
    modelo.train()
    return resultado


torch.manual_seed(0)
sin_atencion = entrenar_juguete(Seq2Seq(VOCAB_JUGUETE, VOCAB_JUGUETE, atencion=False).to(device))
acc_sin = precision_por_largo(sin_atencion)
print({L: f"{a:.0%}" for L, a in acc_sin.items()})

paso   300 · loss 1.012 · 8s


Probemos el modelo con una secuencia corta y una larga:

In [ ]:
def probar(modelo, secuencia):
    src = torch.tensor([[d + 3 for d in secuencia]], device=device)
    pred, _ = modelo.generar(src, torch.tensor([len(secuencia)], device=device), max_largo=len(secuencia))
    esperado = secuencia[::-1]
    obtenido = [t - 3 for t in pred[0].tolist()]
    marcas = "".join("✓" if a == b else "✗" for a, b in zip(obtenido, esperado))
    print("entrada :", secuencia)
    print("esperado:", esperado)
    print("obtenido:", obtenido, marcas, "\n")

probar(sin_atencion, [3, 1, 4, 1, 5])
probar(sin_atencion, [2, 7, 1, 8, 2, 8, 1, 8, 2, 8, 4, 5, 9, 0, 4, 5, 2, 3, 5, 3])

💡 **¿Dónde se equivoca en la secuencia larga?**

Suelen estar hacia el **final** de la salida, que corresponde al **principio** de la entrada: lo primero que leyó el encoder es lo que más se diluyó en el vector de contexto (y además los errores del decoder se acumulan).


---

## 3. Agregar atención

Ahora el decoder, en cada paso, **vuelve a mirar todos los estados del encoder** y elige en cuáles enfocarse:

1. Le da un **puntaje** a cada estado del encoder.
2. Convierte los puntajes en **pesos** que suman 1 → `softmax`.
3. Calcula el **contexto** como el promedio ponderado de los estados del encoder.

Formas: `puntajes` y `pesos` son `(batch, largo_src)`; `enc_salidas` es `(batch, largo_src, d_h)`.

In [ ]:
class AtencionBahdanau(nn.Module):
    def __init__(self, d_h):
        super().__init__()
        self.W = nn.Linear(d_h, d_h, bias=False)
        self.U = nn.Linear(d_h, d_h, bias=False)
        self.v = nn.Linear(d_h, 1, bias=False)

    def forward(self, h, enc_salidas, mascara):
        puntajes = self.v(torch.tanh(self.W(h).unsqueeze(1) + self.U(enc_salidas))).squeeze(-1)
        puntajes = puntajes.masked_fill(~mascara, -1e9)
        pesos = puntajes.softmax(dim=-1)
        contexto = (pesos.unsqueeze(-1) * enc_salidas).sum(dim=1)
        return contexto, pesos

In [ ]:
torch.manual_seed(0)
con_atencion = entrenar_juguete(Seq2Seq(VOCAB_JUGUETE, VOCAB_JUGUETE, atencion=True).to(device))
acc_con = precision_por_largo(con_atencion)

plt.figure(figsize=(7, 4))
plt.plot(list(acc_sin), list(acc_sin.values()), "o-", label="sin atención")
plt.plot(list(acc_con), list(acc_con.values()), "o-", label="con atención")
plt.xlabel("largo de la secuencia"); plt.ylabel("% de tokens correctos")
plt.title("El cuello de botella, medido"); plt.ylim(0, 1.05); plt.grid(alpha=.3); plt.legend(); plt.show()

Es el mismo gráfico de la clase (Bahdanau et al., 2015), ahora con un modelo entrenado aquí. Sin atención, la calidad **cae** con el largo; con atención, se mantiene mucho mejor.

¿Y dónde "mira" el decoder? Dibujemos los pesos de atención para una secuencia:

In [ ]:
def dibujar_atencion(pesos, etiquetas_src, etiquetas_out, titulo=""):
    plt.figure(figsize=(0.45 * len(etiquetas_src) + 2, 0.45 * len(etiquetas_out) + 1.5))
    plt.imshow(pesos, cmap="Purples", vmin=0, vmax=1)
    plt.xticks(range(len(etiquetas_src)), etiquetas_src, rotation=45)
    plt.yticks(range(len(etiquetas_out)), etiquetas_out)
    plt.xlabel("entrada (encoder)"); plt.ylabel("salida (decoder)"); plt.title(titulo)
    plt.colorbar(fraction=0.03); plt.show()

secuencia = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5, 8]
src = torch.tensor([[d + 3 for d in secuencia]], device=device)
pred, pesos = con_atencion.generar(src, torch.tensor([len(secuencia)], device=device), max_largo=len(secuencia))
dibujar_atencion(pesos[0].cpu(), secuencia, [t - 3 for t in pred[0].tolist()], "¿Dónde mira el decoder?")

💡 **¿Por qué la diagonal va "al revés"?** (de arriba a la derecha hacia abajo a la izquierda)

Para escribir el primer token de la salida, la respuesta correcta es el **último** de la entrada; para el segundo, el penúltimo; y así. Nadie le dijo al modelo dónde mirar: **aprendió solo** este alineamiento, igual que el mapa de calor de la clase.

---

## 4. Traducir de verdad: español → inglés

Mismo modelo, datos reales: ~100 mil pares de frases de [Tatoeba](https://tatoeba.org) (el dataset del tutorial de traducción de TensorFlow). Para que entrene rápido usamos solo frases **cortas** (hasta 8 palabras).

### 4.1 Los datos

In [ ]:
import urllib.request, zipfile, io

def descargar_pares():
    try:
        url = "https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
        with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url, timeout=60).read())) as z:
            lineas = z.read("spa-eng/spa.txt").decode("utf-8").strip().split("\n")
        return [tuple(l.split("\t")[:2]) for l in lineas]                      # (inglés, español)
    except Exception as e:
        print("Usando la copia de respaldo en Hugging Face →", e)
        import pandas as pd
        df = pd.read_parquet("https://huggingface.co/datasets/OscarNav/spa-eng/resolve/main/data/train-00000-of-00001.parquet")
        return list(zip(df["eng"], df["spa"]))

pares = descargar_pares()
print(f"{len(pares):,} pares. Algunos ejemplos:")
for en, es in random.sample(pares, 5):
    print(f"  {es}  →  {en}")

Tokenizamos por **palabras** (la Idea 1 del notebook 1), para mantenerlo simple. Las palabras que aparecen una sola vez quedan como `<unk>`.

In [ ]:
def separar_palabras(texto):
    return re.findall(r"\w+|[^\w\s]", texto.lower())

MAX_PALABRAS = 8
vistos, datos_trad = set(), []
for en, es in pares:
    es_t, en_t = separar_palabras(es), separar_palabras(en)
    if len(es_t) <= MAX_PALABRAS and len(en_t) <= MAX_PALABRAS and es not in vistos:   # cortas y sin repetir
        vistos.add(es)
        datos_trad.append((es_t, en_t))
random.shuffle(datos_trad)

UNK = 3
def construir_vocab(frases, min_veces=2):
    conteo = Counter(p for f in frases for p in f)
    vocab = ["[PAD]", "<BOS>", "<EOS>", "<unk>"] + [p for p, n in conteo.most_common() if n >= min_veces]
    return vocab, {p: i for i, p in enumerate(vocab)}

vocab_es, id_es = construir_vocab([es for es, en in datos_trad])
vocab_en, id_en = construir_vocab([en for es, en in datos_trad])
print(f"{len(datos_trad):,} pares cortos · vocabulario: {len(vocab_es):,} palabras en español, {len(vocab_en):,} en inglés")

In [ ]:
def a_tensores(lote):
    largos = torch.tensor([len(es) for es, en in lote])
    src = torch.full((len(lote), int(largos.max())), PAD)
    tgt = torch.full((len(lote), MAX_PALABRAS + 2), PAD)
    for i, (es, en) in enumerate(lote):
        src[i, :len(es)] = torch.tensor([id_es.get(p, UNK) for p in es])
        ids_en = [BOS] + [id_en.get(p, UNK) for p in en] + [EOS]
        tgt[i, :len(ids_en)] = torch.tensor(ids_en)
    return src.to(device), tgt.to(device), largos.to(device)

prueba, entrenamiento = datos_trad[:500], datos_trad[500:]
src, tgt, largos = a_tensores(entrenamiento[:2])
print(entrenamiento[0], "\n→ src:", src[0].tolist(), "\n→ tgt:", tgt[0].tolist())

### 4.2 Entrenar

Usamos **las mismas clases** `Seq2Seq`, `Encoder`, `Decoder` y `AtencionBahdanau` de la tarea de juguete; solo cambian los vocabularios y el tamaño. 5 pasadas por los datos.

In [ ]:
def entrenar_traductor(modelo, epocas=5, batch=64):
    opt = torch.optim.Adam(modelo.parameters(), lr=2e-3)
    inicio = time.time()
    for epoca in range(1, epocas + 1):
        random.shuffle(entrenamiento)
        total = 0
        for i in range(0, len(entrenamiento), batch):
            src, tgt, largos = a_tensores(entrenamiento[i:i + batch])
            logits = modelo(src, largos, tgt)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1), ignore_index=PAD)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            opt.step()
            total += loss.item()
        print(f"época {epoca} · loss {total / (len(entrenamiento) / batch):.2f} · {time.time() - inicio:.0f}s")
    return modelo

torch.manual_seed(0)
traductor = entrenar_traductor(Seq2Seq(len(vocab_es), len(vocab_en), atencion=True, d_emb=128, d_h=256).to(device))

In [ ]:
def traducir(frase, mostrar_atencion=False):
    palabras = separar_palabras(frase)
    src = torch.tensor([[id_es.get(p, UNK) for p in palabras]], device=device)
    pred, pesos = traductor.generar(src, torch.tensor([len(palabras)], device=device), max_largo=MAX_PALABRAS + 2)
    salida = []
    for t in pred[0].tolist():
        if t == EOS:
            break
        salida.append(vocab_en[t])
    vistas = [p if p in id_es else f"{p}→<unk>" for p in palabras]
    print(f"{' '.join(vistas)}  →  {' '.join(salida)}")
    if mostrar_atencion:
        dibujar_atencion(pesos[0, :len(salida) + 1].cpu(), palabras, salida + ["<EOS>"], "Atención: español → inglés")

for es, en in prueba[:8]:
    traducir(" ".join(es))
    print(f"   (referencia: {' '.join(en)})\n")

Y nuestra frase de la clase, con su mapa de atención:

In [ ]:
traducir("no me gusta el café .", mostrar_atencion=True)

Es el mismo tipo de mapa de calor de la clase: cada palabra en inglés "mira" sobre todo a la palabra en español que le corresponde.

### Más frases

Se pueden probar frases que propongan en el chat (cortas, hasta 8 palabras). Conviene mostrar una que salga bien, una que salga mal y una con una palabra que el modelo no conoce (aparece como `→<unk>`).

In [ ]:
traducir("tengo un perro .")
traducir("¿dónde está el baño ?")
traducir("mañana vamos a la playa .")
# traducir("frase del chat .")

💡 **Qué se ve**

- Con frases cortas y comunes funciona sorprendentemente bien para un modelo tan chico.
- Falla con palabras raras, con frases largas (solo vio hasta 8 palabras) y con cosas ambiguas.
- Las palabras desconocidas son `<unk>`: el modelo no tiene **ninguna** información sobre ellas. Es exactamente el problema del Acto 1. Con **subpalabras** (BPE), "googleé" se armaría con piezas conocidas en vez de perderse.

Los traductores modernos usan subpalabras + **Transformers** (solo atención, sin recurrencia) y miles de millones de frases. Eso es lo que viene en las próximas clases.


---

## ✅ Resumen

| | Lo que vimos |
|---|---|
| Encoder–decoder | El encoder resume la entrada en un vector; el decoder escribe de a un token (teacher forcing al entrenar) |
| Cuello de botella | **Medimos** cómo cae la calidad con el largo cuando todo pasa por un solo vector |
| Atención | Programada a mano: puntajes → softmax → promedio ponderado. Viste la diagonal (y la anti-diagonal) |
| Traducción | Un traductor español → inglés real, con las mismas piezas |

🎯 **La conexión con lo que viene:** un Transformer usa **tokens de subpalabras** (notebook 1), se entrena **prediciendo el siguiente token** (notebook 2) y su corazón es la **atención** (este notebook), pero sin la RNN.